# Plot results
- Note: all the input data for this notebook is available in Data

In [ ]:
import pandas as pd
import os
import numpy as np
import re
import ast, json
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.patches import Patch
from scipy.stats import ttest_ind
from statsmodels.stats.multitest import multipletests
from scipy.stats import ttest_ind
from statsmodels.stats.multitest import multipletests
from statannotations.Annotator import Annotator
from matplotlib.ticker import MaxNLocator
from scipy.stats import t as t_dist

In [ ]:
# Output directory, create if doesnt exist
os.makedirs("Output", exist_ok=True)

### GVT (truth) data

In [ ]:
# generated by running https://github.com/uc-cdis/gdc-qag/blob/master/truth.py code
# truth contains frequencies from the GDC for each question
truth_data = pd.read_csv('Data/truth.csv', index_col=0)

In [ ]:
truth_data.head(n=3)

In [ ]:
truth_data.shape

### gpt-oss-20b

In [ ]:
gpt_oss_20b_results = pd.read_csv('Data/gpt_oss_20b_baseline.csv', index_col=0)

In [ ]:
gpt_oss_20b_results.head(n=2)

In [ ]:
gpt_oss_20b_results.shape

In [ ]:
### write a function to extract just base stat from the response above
def extract_percentage(text):
    match = re.search(r'([\d.]+)%', str(text))
    return float(match.group(1))


In [ ]:
gpt_oss_20b_results_join_truth = pd.merge(
    left = gpt_oss_20b_results,
    right = truth_data,
    on = ['questions'],
)

In [ ]:
gpt_oss_20b_results_join_truth['gpt_oss_20b_base_stat'] = gpt_oss_20b_results_join_truth['gpt_oss_20b_base_output'].apply(extract_percentage)

In [ ]:
gpt_oss_20b_results_join_truth.head(n=3)

In [ ]:
gpt_oss_20b_results_join_truth['delta_gpt_oss_20b'] = gpt_oss_20b_results_join_truth['gpt_oss_20b_base_stat'] - gpt_oss_20b_results_join_truth['truth']

In [ ]:
gpt_oss_20b_results_join_truth.shape

In [ ]:
gpt_oss_20b_results_join_truth['intent'].value_counts()

In [ ]:
# top_cases_counts_by_gene questions are cnv_and_ssm type
# merge into one for plotting questions of this type
gpt_oss_20b_results_join_truth.loc[gpt_oss_20b_results_join_truth['intent'] == 'top_cases_counts_by_gene', 'intent'] = 'cnv_and_ssm'

In [ ]:
gpt_oss_20b_results_join_truth['intent'].value_counts()

In [ ]:
gpt_oss_20b_results_join_truth['delta_gpt_oss_20b'].describe()

In [ ]:
gpt_oss_20b_results_join_truth.groupby('intent')['delta_gpt_oss_20b'].describe()

In [ ]:
gpt_oss_20b_results_join_truth['delta_gpt_oss_20b'].describe().to_frame('ALL (overall)').T

In [ ]:
col = 'delta_gpt_oss_20b'

by_intent = gpt_oss_20b_results_join_truth.groupby('intent')[col].describe()
overall = gpt_oss_20b_results_join_truth[col].describe().to_frame('ALL (overall)').T

summary = pd.concat([by_intent, overall])
summary.to_csv('Output/gpt_oss_20b.baseline_stats.csv', index_label='intent')
summary

### Distribution of deltas (supplementary histograms)

In [ ]:
# we will use one panel per intent, colored by model

def delta_hist(df, col, color, model_label, fname, ncols=3):
    # number of intent groups, which is 5 for us
    groups = list(df.groupby('intent'))
    # for plotting, how many rows and cols to use
    nrows = int(np.ceil(len(groups) / ncols))
    bins = np.linspace(-100, 100, 41)
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3 * nrows), sharex=True, sharey=False, squeeze=False)
    lo, hi = df[col].min(), df[col].max()
    pad = 5
    for ax, (name, group) in zip(axes.flat, groups):
        ax.hist(group[col], bins=bins, color=color, edgecolor='black', linewidth=0.4)
        ax.axvline(0, color='black', linestyle='--', lw=1)
        ax.set_title('%s (n=%d)' % (name, len(group)), fontsize=8, fontweight='bold')
        ax.tick_params(labelsize=8)
        ax.yaxis.set_major_locator(MaxNLocator(integer=True))  # per-panel y range, whole-number counts
        # x-limits
        ax.set_xlim(lo - pad, hi + pad)
    # hide unused panels in the grid, e.g. the 6th one since only 5 intents
    for ax in axes.flat[len(groups):]:
        ax.set_visible(False)
    fig.supxlabel('Freq (Model) % - Freq (GDC) %', fontweight='bold')
    fig.supylabel('Number of queries', fontweight='bold')
    fig.suptitle(model_label, fontweight='bold')
    plt.tight_layout()
    plt.savefig(fname, bbox_inches='tight', dpi=300)

delta_hist(gpt_oss_20b_results_join_truth, 'delta_gpt_oss_20b', 'lightgreen', 'gpt-oss-20b', 'figs/S1.png')


### gpt-oss-20b with MCP

In [ ]:
gpt_oss_20b_with_mcp = pd.read_csv('Data/gpt_oss_20b_with_mcp.csv')

In [ ]:
gpt_oss_20b_with_mcp.head(n=3)

In [ ]:
gpt_oss_20b_with_mcp.shape

In [ ]:
gpt_oss_20b_with_mcp['intent'].unique()

In [ ]:
gpt_oss_20b_with_mcp['truth'].describe()

In [ ]:
# compute difference: answer_average - truth
# each query tried 3x, and answer_average has the average frequencies
gpt_oss_20b_with_mcp['delta_gpt_oss_20b_with_mcp'] = gpt_oss_20b_with_mcp['answer_average'] - gpt_oss_20b_with_mcp['truth']


In [ ]:
gpt_oss_20b_with_mcp['delta_gpt_oss_20b_with_mcp'].describe()

In [ ]:
gpt_oss_20b_with_mcp.loc[gpt_oss_20b_with_mcp['intent'] == 'top_cases_counts_by_gene', 'intent'] = 'cnv_and_ssm'

In [ ]:
gpt_oss_20b_with_mcp['intent'].value_counts()

In [ ]:
gpt_oss_20b_with_mcp.groupby('intent')['delta_gpt_oss_20b_with_mcp'].describe()

In [ ]:
col = 'delta_gpt_oss_20b_with_mcp'

by_intent = gpt_oss_20b_with_mcp.groupby('intent')[col].describe()
overall = gpt_oss_20b_with_mcp[col].describe().to_frame('ALL (overall)').T

summary = pd.concat([by_intent, overall])
summary.to_csv('Output/gpt_oss_20b.with_mcp.baseline_stats.csv', index_label='intent')
summary

In [ ]:
delta_hist(gpt_oss_20b_with_mcp, 'delta_gpt_oss_20b_with_mcp', 'darkgreen', 'gpt-oss-20b+MCP', 'figs/S5.png')


### gpt-oss-120b

In [ ]:
gpt_oss_120b_results = pd.read_csv('Data/gpt_oss_120b_baseline.csv', index_col=0)

In [ ]:
gpt_oss_120b_results.head(n=2)

In [ ]:
gpt_oss_120b_results_join_truth = pd.merge(
    left = gpt_oss_120b_results,
    right = truth_data,
    on = ['questions'],
)

In [ ]:
gpt_oss_120b_results_join_truth['gpt_oss_120b_base_stat'] = gpt_oss_120b_results_join_truth['gpt_oss_120b_base_output'].apply(extract_percentage)

In [ ]:
gpt_oss_120b_results_join_truth['delta_gpt_oss_120b'] = gpt_oss_120b_results_join_truth['gpt_oss_120b_base_stat'] - gpt_oss_120b_results_join_truth['truth']

In [ ]:
gpt_oss_120b_results_join_truth.loc[gpt_oss_120b_results_join_truth['intent'] == 'top_cases_counts_by_gene', 'intent'] = 'cnv_and_ssm'

In [ ]:
gpt_oss_120b_results_join_truth['intent'].value_counts()

In [ ]:
gpt_oss_120b_results_join_truth['delta_gpt_oss_120b'].describe()

In [ ]:
gpt_oss_120b_results_join_truth.groupby('intent')['delta_gpt_oss_120b'].describe()

In [ ]:
col = 'delta_gpt_oss_120b'

by_intent = gpt_oss_120b_results_join_truth.groupby('intent')[col].describe()
overall = gpt_oss_120b_results_join_truth[col].describe().to_frame('ALL (overall)').T

summary = pd.concat([by_intent, overall])
summary.to_csv('Output/gpt_oss_120b.baseline_stats.csv', index_label='intent')
summary

In [ ]:
delta_hist(gpt_oss_120b_results_join_truth, 'delta_gpt_oss_120b', 'lightcoral', 'gpt-oss-120b', 'figs/S2.png')


### gpt-oss-120b + MCP

In [ ]:
gpt_oss_120b_with_mcp = pd.read_csv('Data/gpt_oss_120b_with_mcp.csv')

In [ ]:
gpt_oss_120b_with_mcp.head(n=3)

In [ ]:
# recompute difference as answer_average - truth
gpt_oss_120b_with_mcp['delta_gpt_oss_120b_with_mcp'] = gpt_oss_120b_with_mcp['answer_average'] - gpt_oss_120b_with_mcp['truth']


In [ ]:
gpt_oss_120b_with_mcp['delta_gpt_oss_120b_with_mcp'].describe()

In [ ]:
gpt_oss_120b_with_mcp.loc[gpt_oss_120b_with_mcp['intent'] == 'top_cases_counts_by_gene', 'intent'] = 'cnv_and_ssm'

In [ ]:
gpt_oss_120b_with_mcp['intent'].value_counts()

In [ ]:
gpt_oss_120b_with_mcp.groupby('intent')['delta_gpt_oss_120b_with_mcp'].describe()

In [ ]:
col = 'delta_gpt_oss_120b_with_mcp'

by_intent = gpt_oss_120b_with_mcp.groupby('intent')[col].describe()
overall = gpt_oss_120b_with_mcp[col].describe().to_frame('ALL (overall)').T

summary = pd.concat([by_intent, overall])
summary.to_csv('Output/gpt_oss_120b.with_mcp.baseline_stats.csv', index_label='intent')
summary

In [ ]:
gpt_oss_120b_with_mcp.shape

In [ ]:
delta_hist(gpt_oss_120b_with_mcp, 'delta_gpt_oss_120b_with_mcp', 'firebrick', 'gpt-oss-120b+MCP', 'figs/S6.png')


## Combine all data for plotting
- gpt_oss_20b_results_join_truth
- gpt_oss_120b_results_join_truth
- gpt_oss_20b_with_mcp
- gpt_oss_120b_with_mcp

### Plot with significance values

In [ ]:
# spacing parameters
line_height = 2
text_offset = 0.5

fig, ax = plt.subplots(figsize=(16, 8))

# list of delta values for each intent
grouped_oss20b_data = gpt_oss_20b_results_join_truth.groupby('intent')['delta_gpt_oss_20b'].apply(list)
grouped_oss120b_data = gpt_oss_120b_results_join_truth.groupby('intent')['delta_gpt_oss_120b'].apply(list)
grouped_oss20b_with_mcp_data = gpt_oss_20b_with_mcp.groupby('intent')['delta_gpt_oss_20b_with_mcp'].apply(list)
grouped_oss120b_with_mcp_data = gpt_oss_120b_with_mcp.groupby('intent')['delta_gpt_oss_120b_with_mcp'].apply(list)

# n = number of intent groups
n = len(grouped_oss20b_data)
# xlabels = intent names
xlabels = list(grouped_oss20b_data.groupby('intent').groups.keys())

# define the order of positions for plotting
# this is the order in which it will appear on the plot
positions_oss20b = np.arange(n) * 4
positions_oss20b_with_mcp = positions_oss20b + 0.8
positions_oss120b = positions_oss20b_with_mcp + 0.8
positions_oss120b_with_mcp = positions_oss120b + 0.8

ax.boxplot(
    grouped_oss20b_data, positions=positions_oss20b,
    notch=True, 
    flierprops=dict(marker='o', markerfacecolor='lightgreen', markeredgecolor='lightgreen', markersize=3, linestyle='none'), 
    patch_artist=True, 
    boxprops=dict(facecolor='lightgreen'),
    showmeans=False
)
ax.boxplot(
    grouped_oss20b_with_mcp_data, positions=positions_oss20b_with_mcp,
    notch=True, 
    flierprops=dict(marker='o', markerfacecolor='darkgreen', markeredgecolor='darkgreen', markersize=3, linestyle='none'), 
    patch_artist=True, 
    boxprops=dict(facecolor='darkgreen'),
    showmeans=False
)
ax.boxplot(
    grouped_oss120b_data, positions=positions_oss120b,
    notch=True, 
    flierprops=dict(marker='o', markerfacecolor='lightcoral', markeredgecolor='lightcoral', markersize=3, linestyle='none'), 
    patch_artist=True, 
    boxprops=dict(facecolor='lightcoral'),
    showmeans=False
)
ax.boxplot(
    grouped_oss120b_with_mcp_data, positions=positions_oss120b_with_mcp,
    notch=True, 
    flierprops=dict(marker='o', markerfacecolor='firebrick', markeredgecolor='firebrick', markersize=3, linestyle='none'), 
    patch_artist=True, 
    boxprops=dict(facecolor='firebrick'),
    showmeans=False
)

# x-axis ticks
# get the positions to place the labels
# all_positions is the center position of each group, one for each of the 5 intents
all_positions = (positions_oss20b + positions_oss20b_with_mcp + positions_oss120b + positions_oss120b_with_mcp) / 4
ax.set_xticks(all_positions)
ax.set_xticklabels(xlabels, fontsize=8, rotation=90, fontweight='bold')

# yticks
yticks = np.arange(-100,100,10)
ax.set_yticks(yticks)
ytick_labels = [str(t) if t % 20 == 0 else '' for t in yticks]
ax.set_yticklabels(ytick_labels)

# ylabel
ax.set_ylabel('Freq (Model) % - Freq (GDC) %', fontweight='bold')
ax.axhline(y=0, color='black', linestyle='--')


# p-values: three comparisons for each intent
# gpt-oss-20b +/- MCP 
# gpt-oss-120b +/- MCP
# gpt-oss-20b-with-mcp vs gpt-oss-120b-with-mcp
comparisons = []
# for each intent, get the list of frequency values
for i, intent in enumerate(xlabels):
    oss20b_vals      = grouped_oss20b_data[intent]
    oss20b_mcp_vals  = grouped_oss20b_with_mcp_data[intent]
    oss120b_vals     = grouped_oss120b_data[intent]
    oss120b_mcp_vals = grouped_oss120b_with_mcp_data[intent]
    # to set the height where significance values are drawn, determine max value
    top = max(max(oss20b_vals), max(oss20b_mcp_vals), max(oss120b_vals), max(oss120b_mcp_vals))

    # set some y_offsets to show the significance stars and brackets
    for (vals_a, vals_b, y_offset, x1_pos, x2_pos) in [
        (oss20b_vals,     oss20b_mcp_vals,  2, positions_oss20b[i],          positions_oss20b_with_mcp[i]),   # 20b vs 20b+MCP
        (oss120b_vals,    oss120b_mcp_vals, 2, positions_oss120b[i],         positions_oss120b_with_mcp[i]),  # 120b vs 120b+MCP
        (oss20b_mcp_vals, oss120b_mcp_vals, 12, positions_oss20b_with_mcp[i], positions_oss120b_with_mcp[i]),  # 20b+MCP vs 120b+MCP
    ]:
        _, p = ttest_ind(vals_a, vals_b, equal_var=False)
        comparisons.append((top + y_offset, x1_pos, x2_pos, p))

# FDR-BH correction across all tests (3 per intent)
_, corrected_p, _, _ = multipletests([c[3] for c in comparisons], method='fdr_bh')

# plot brackets with corrected p-values
for (y, x1, x2, _), p in zip(comparisons, corrected_p):
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
    ax.plot([x1, x1, x2, x2], [y, y + line_height, y + line_height, y], color='black', lw=1)
    ax.text((x1 + x2) / 2, y + line_height + text_offset, sig, ha='center', va='bottom', fontsize=6)

# Legend
legend_elements = [
    Patch(facecolor='lightgreen', label='gpt-oss-20b'),
    Patch(facecolor='darkgreen',  label='gpt-oss-20b+MCP'),
    Patch(facecolor='lightcoral', label='gpt-oss-120b'),
    Patch(facecolor='firebrick',     label='gpt-oss-120b+MCP'),
]

ax.legend(bbox_to_anchor=(1, 1), loc='upper left', fontsize=8, handles=legend_elements)
plt.tight_layout()
plt.title('')
plt.savefig('figs/Fig2A.png', bbox_inches='tight', dpi=300)

### separate heterozygous deletions from gains and amplifications

In [ ]:
def return_cnv_type(x):
    if 'heterozygous deletion' in x:
        cnv_type = 'heterozygous deletion'
    elif 'gain' in x:
        cnv_type = 'gain'
    elif 'amplification' in x:
        cnv_type = 'amplification'
    else:
        cnv_type = 'NA'
    return cnv_type

In [ ]:
gpt_oss_20b_results_join_truth['cnv_type'] = gpt_oss_20b_results_join_truth['questions'].apply(return_cnv_type)
gpt_oss_20b_with_mcp['cnv_type'] = gpt_oss_20b_with_mcp['questions'].apply(return_cnv_type)
gpt_oss_120b_results_join_truth['cnv_type'] = gpt_oss_120b_results_join_truth['questions'].apply(return_cnv_type)
gpt_oss_120b_with_mcp['cnv_type'] = gpt_oss_120b_with_mcp['questions'].apply(return_cnv_type)


In [ ]:
gpt_oss_20b_cnv = gpt_oss_20b_results_join_truth[gpt_oss_20b_results_join_truth['intent'].str.contains('freq_cnv_loss')]
gpt_oss_20b_with_mcp_cnv = gpt_oss_20b_with_mcp[gpt_oss_20b_with_mcp['intent'].str.contains('freq_cnv_loss')]
gpt_oss_120b_cnv = gpt_oss_120b_results_join_truth[gpt_oss_120b_results_join_truth['intent'].str.contains('freq_cnv_loss')]
gpt_oss_120b_with_mcp_cnv = gpt_oss_120b_with_mcp[gpt_oss_120b_with_mcp['intent'].str.contains('freq_cnv_loss')]

In [ ]:
gpt_oss_20b_cnv.head(n=3)

In [ ]:
# the two NAs are for the two homozygous deletion questions
gpt_oss_20b_cnv['cnv_type'].value_counts()

In [ ]:
# prepare data
gpt_oss_20b_cnv['model'] = 'gpt-oss-20b'
gpt_oss_120b_cnv['model'] = 'gpt-oss-120b'
gpt_oss_20b_with_mcp_cnv['model'] = 'gpt-oss-20b-with-mcp'
gpt_oss_120b_with_mcp_cnv['model'] = 'gpt-oss-120b-with-mcp'

In [ ]:
# 20b vs 20b+MCP, 120b vs 120b+MCP, 20b+MCP vs 120b+MCP per CNV type, with FDR correction
# spacing parameters
line_height = 2
text_offset = 0.5

fig, ax = plt.subplots(figsize=(10, 8))

cnv_order = ['amplification', 'gain', 'heterozygous deletion']
cnvs = {
    '20b':      (gpt_oss_20b_cnv,          'delta_gpt_oss_20b'),
    '20b_mcp':  (gpt_oss_20b_with_mcp_cnv, 'delta_gpt_oss_20b_with_mcp'),
    '120b':     (gpt_oss_120b_cnv,         'delta_gpt_oss_120b'),
    '120b_mcp': (gpt_oss_120b_with_mcp_cnv,'delta_gpt_oss_120b_with_mcp'),
}

# for each cnv_type get the list of delta values for each of the four models
grouped = {}
for k, (df, col) in cnvs.items():
    df = df[df['cnv_type'].isin(cnv_order)]
    grouped[k] = df.groupby('cnv_type')[col].apply(list)

n = len(cnv_order)
xlabels = cnv_order
positions_oss20b = np.arange(n) * 4
positions_oss20b_with_mcp = positions_oss20b + 0.8
positions_oss120b = positions_oss20b_with_mcp + 0.8
positions_oss120b_with_mcp = positions_oss120b + 0.8

box_colors = [
    ('20b',      positions_oss20b,          'lightgreen'),
    ('20b_mcp',  positions_oss20b_with_mcp, 'darkgreen'),
    ('120b',     positions_oss120b,         'lightcoral'),
    ('120b_mcp', positions_oss120b_with_mcp,'firebrick'),
]
for key, pos, color in box_colors:
    ax.boxplot(
        grouped[key].loc[cnv_order], positions=pos,
        notch=True,
        flierprops=dict(marker='o', markerfacecolor=color, markeredgecolor=color, markersize=3, linestyle='none'),
        patch_artist=True, boxprops=dict(facecolor=color), showmeans=False
    )

# x-axis ticks
all_positions = (positions_oss20b + positions_oss20b_with_mcp + positions_oss120b + positions_oss120b_with_mcp) / 4
ax.set_xticks(all_positions)
ax.set_xticklabels(xlabels, fontsize=8, fontweight='bold')

# yticks
yticks = np.arange(-100, 100, 10)
ax.set_yticks(yticks)
ax.set_yticklabels([str(t) if t % 20 == 0 else '' for t in yticks])
ax.set_ylabel('Freq (Model) % - Freq (GDC) %', fontweight='bold')
ax.axhline(y=0, color='black', linestyle='--')

# p-values: 3 comparisons per CNV type
comparisons = []
for i, cnv_type in enumerate(xlabels):
    oss20b_vals      = grouped['20b'][cnv_type]
    oss20b_mcp_vals  = grouped['20b_mcp'][cnv_type]
    oss120b_vals     = grouped['120b'][cnv_type]
    oss120b_mcp_vals = grouped['120b_mcp'][cnv_type]
    top = max(max(oss20b_vals), max(oss20b_mcp_vals), max(oss120b_vals), max(oss120b_mcp_vals))

    for (vals_a, vals_b, y_offset, x1_pos, x2_pos) in [
        (oss20b_vals,     oss20b_mcp_vals,  2, positions_oss20b[i],          positions_oss20b_with_mcp[i]),   # 20b vs 20b+MCP
        (oss120b_vals,    oss120b_mcp_vals, 2, positions_oss120b[i],         positions_oss120b_with_mcp[i]),  # 120b vs 120b+MCP
        (oss20b_mcp_vals, oss120b_mcp_vals, 12, positions_oss20b_with_mcp[i], positions_oss120b_with_mcp[i]), # 20b+MCP vs 120b+MCP
    ]:
        _, p = ttest_ind(vals_a, vals_b, equal_var=False)
        comparisons.append((top + y_offset, x1_pos, x2_pos, p))

# FDR-BH correction across all 9 tests
_, corrected_p, _, _ = multipletests([c[3] for c in comparisons], method='fdr_bh')

# plot brackets with corrected p-values
for (y, x1, x2, _), p in zip(comparisons, corrected_p):
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
    ax.plot([x1, x1, x2, x2], [y, y + line_height, y + line_height, y], color='black', lw=1)
    ax.text((x1 + x2) / 2, y + line_height + text_offset, sig, ha='center', va='bottom', fontsize=6)

# Legend
legend_elements = [
    Patch(facecolor='lightgreen', label='gpt-oss-20b'),
    Patch(facecolor='darkgreen',  label='gpt-oss-20b+MCP'),
    Patch(facecolor='lightcoral', label='gpt-oss-120b'),
    Patch(facecolor='firebrick',  label='gpt-oss-120b+MCP'),
]
ax.legend(bbox_to_anchor=(1, 1), loc='upper left', fontsize=8, handles=legend_elements)
plt.tight_layout()
plt.title('')
plt.savefig('figs/Fig2B.png', bbox_inches='tight', dpi=300)

### Separate TCGA and non-TCGA questions
- gpt_oss_20b_results_join_truth
- gpt_oss_120b_results_join_truth
- gpt_oss_20b_with_mcp
- gpt_oss_120b_with_mcp (delta_gpt_oss_120b_with_mcp)

In [ ]:
def return_tcga_label(x):
    if 'TCGA' in x['questions']:
        return 'TCGA'
    else:
        return 'Non-TCGA'

In [ ]:
gpt_oss_20b_results_join_truth['is_tcga'] = gpt_oss_20b_results_join_truth.apply(
    lambda x: return_tcga_label(x), axis=1
)

In [ ]:
gpt_oss_120b_results_join_truth['is_tcga'] = gpt_oss_120b_results_join_truth.apply(
    lambda x: return_tcga_label(x), axis=1
)

In [ ]:
gpt_oss_20b_with_mcp['is_tcga'] = gpt_oss_20b_with_mcp.apply(
    lambda x: return_tcga_label(x), axis=1
)

In [ ]:
gpt_oss_120b_with_mcp['is_tcga'] = gpt_oss_120b_with_mcp.apply(
    lambda x: return_tcga_label(x), axis=1
)

### show TCGA and non-TCGA performance side-by-side for each LLM

In [ ]:
# create TCGA vs non-TCGA boxplots, colored by model
def tcga_boxplot(df, col, color, model_label, fname):
    fig, ax = plt.subplots(figsize=(10, 6))
    rng = np.random.default_rng(0)
    intents = sorted(df['intent'].unique())
    line_height, text_offset, buffer_above_box = 2, 0.5, 2

    # TCGA box (filled) next to non-TCGA box (outline only) for each intent
    vals, pos = {}, {}
    for i, intent in enumerate(intents):
        for k, grp in enumerate(['TCGA', 'Non-TCGA']):
            v = df.loc[(df['intent'] == intent) & (df['is_tcga'] == grp), col].tolist()
            if not v:
                continue
            # tick position for TCGA/non-TCGA
            vals[(intent, grp)], pos[(intent, grp)] = v, i * 4 + k * 1.2
            if grp == 'TCGA':
                boxprops = dict(facecolor=color, edgecolor='black', alpha=0.6)
            else:
                boxprops = dict(facecolor='none', edgecolor=color, linewidth=1.5)
            ax.boxplot(v, positions=[pos[(intent, grp)]], widths=0.9, notch=True, showfliers=False,
                       patch_artist=True, showmeans=False, boxprops=boxprops,
                       medianprops=dict(color='black', linewidth=1.5))
            # jittered points instead of fliers: filled for TCGA, hollow for non-TCGA
            x = pos[(intent, grp)] + rng.uniform(-0.3, 0.3, len(v))
            ax.scatter(x, v, s=6, alpha=0.4, zorder=3,
                       facecolors=color if grp == 'TCGA' else 'none', edgecolors=color, linewidths=0.6)
    # x-axis ticks centered on each intent pair
    ax.set_xticks([i * 4 + 0.6 for i in range(len(intents))])
    ax.set_xticklabels(intents, fontsize=8, rotation=90, fontweight='bold')

    # y-axis ticks
    yticks = np.arange(-100, 100, 10)
    ax.set_yticks(yticks)
    ax.set_yticklabels([str(t) if t % 20 == 0 else '' for t in yticks])
    ax.set_ylabel('Freq (%s) %% - Freq (GDC) %%' % model_label, fontweight='bold')
    ax.axhline(y=0, color='black', linestyle='--')

    # p-values: TCGA vs non-TCGA per intent, FDR-BH corrected
    comparisons = []
    for intent in intents:
        if (intent, 'TCGA') in vals and (intent, 'Non-TCGA') in vals:
            a, b = vals[(intent, 'TCGA')], vals[(intent, 'Non-TCGA')]
            _, p = ttest_ind(a, b, equal_var=False)
            comparisons.append((max(max(a), max(b)), pos[(intent, 'TCGA')], pos[(intent, 'Non-TCGA')], p))
    _, corrected_p, _, _ = multipletests([c[3] for c in comparisons], method='fdr_bh')
    for (top, x1, x2, _), p in zip(comparisons, corrected_p):
        sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
        y = top + buffer_above_box
        ax.plot([x1, x1, x2, x2], [y, y + line_height, y + line_height, y], color='black', lw=1)
        ax.text((x1 + x2) / 2, y + line_height + text_offset, sig, ha='center', va='bottom', fontsize=6)

    legend_elements = [
        Patch(facecolor=color, edgecolor='black', label='TCGA'),
        Patch(facecolor='none', edgecolor=color, label='Non-TCGA'),
    ]
    ax.legend(handles=legend_elements, bbox_to_anchor=(1, 1), loc='upper left', fontsize=8)
    plt.tight_layout()
    plt.title('')
    plt.savefig(fname, bbox_inches='tight', dpi=300)

# oss-20b
tcga_boxplot(gpt_oss_20b_results_join_truth, 'delta_gpt_oss_20b', 'lightgreen', 'gpt-oss-20b', 'figs/S3.png')


In [ ]:
# oss-20b+MCP
tcga_boxplot(gpt_oss_20b_with_mcp, 'delta_gpt_oss_20b_with_mcp', 'darkgreen', 'gpt-oss-20b+MCP', 'figs/S7.png')


In [ ]:
# oss-120b
tcga_boxplot(gpt_oss_120b_results_join_truth, 'delta_gpt_oss_120b', 'lightcoral', 'gpt-oss-120b', 'figs/S4.png')


In [ ]:
# oss-120b+MCP
tcga_boxplot(gpt_oss_120b_with_mcp, 'delta_gpt_oss_120b_with_mcp', 'firebrick', 'gpt-oss-120b+MCP', 'figs/S8.png')


### Overall mean difference from GDC for each model
- show mean across all intents
- overall mean per intent
- show error bars: 95% CI of the mean

In [ ]:
models = {
    'gpt-oss-20b':      (gpt_oss_20b_results_join_truth, 'delta_gpt_oss_20b',          'lightgreen'),
    'gpt-oss-20b+MCP':  (gpt_oss_20b_with_mcp,           'delta_gpt_oss_20b_with_mcp', 'darkgreen'),
    'gpt-oss-120b':     (gpt_oss_120b_results_join_truth,'delta_gpt_oss_120b',         'lightcoral'),
    'gpt-oss-120b+MCP': (gpt_oss_120b_with_mcp,          'delta_gpt_oss_120b_with_mcp','firebrick'),
}


In [ ]:
[df[col].mean() for df, col, _ in models.values()]

In [ ]:
# mean difference from GDC per model: 
# across all intents; per intent (error bars = 95% CI of the mean)
models = {
    'gpt-oss-20b':      (gpt_oss_20b_results_join_truth, 'delta_gpt_oss_20b',          'lightgreen'),
    'gpt-oss-20b+MCP':  (gpt_oss_20b_with_mcp,           'delta_gpt_oss_20b_with_mcp', 'darkgreen'),
    'gpt-oss-120b':     (gpt_oss_120b_results_join_truth,'delta_gpt_oss_120b',         'lightcoral'),
    'gpt-oss-120b+MCP': (gpt_oss_120b_with_mcp,          'delta_gpt_oss_120b_with_mcp','firebrick'),
}

# calculate 95% CI of the mean
def ci95(x):
    x = np.asarray(x, dtype=float)
    x = x[~np.isnan(x)]
    return t_dist.ppf(0.975, len(x) - 1) * x.std(ddof=1) / np.sqrt(len(x)) if len(x) > 1 else np.nan

labels = list(models)
colors = [c for _, _, c in models.values()]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), sharey=True, gridspec_kw={'width_ratios': [1, 2.5]})

# mean delta_* for all queries, i.e across all intents 
means = [df[col].mean() for df, col, _ in models.values()]
# 95% CI of the mean
cis  = [ci95(df[col]) for df, col, _ in models.values()]
x = np.arange(len(labels))
ax1.bar(x, means, yerr=cis, color=colors, edgecolor='black', linewidth=0.6, capsize=4, width=0.7)
# value labels just beyond the end of each error bar
for xi, m, ci_val in zip(x, means, cis):
    ax1.text(xi, (m - ci_val - 0.3) if m < 0 else (m + ci_val + 0.3), f'{m:.2f}',
             ha='center', va='top' if m < 0 else 'bottom', fontsize=7, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(labels, fontsize=8, rotation=90, fontweight='bold')
ax1.set_ylabel('Mean Freq (Model) % - Freq (GDC) %', fontweight='bold')
ax1.set_title('All intents', fontsize=10, fontweight='bold')

# per intent
intents = sorted(gpt_oss_20b_results_join_truth['intent'].unique())
width = 0.2
xi = np.arange(len(intents))
for k, (label, (df, col, color)) in enumerate(models.items()):
    g = df.groupby('intent')[col].agg(['mean', ci95]).reindex(intents)
    ax2.bar(xi + (k - 1.5) * width, g['mean'], width, yerr=g['ci95'], color=color,
            edgecolor='black', linewidth=0.6, capsize=2, error_kw=dict(lw=0.8))
ax2.set_xticks(xi)
ax2.set_xticklabels(intents, fontsize=8, rotation=90, fontweight='bold')
ax2.set_title('Per intent', fontsize=10, fontweight='bold')

for ax in (ax1, ax2):
    ax.axhline(y=0, color='black', linestyle='--')

legend_elements = [Patch(facecolor=c, edgecolor='black', label=l) for l, c in zip(labels, colors)]
ax2.legend(handles=legend_elements, bbox_to_anchor=(1, 1), loc='upper left', fontsize=8)

plt.tight_layout()
plt.savefig('figs/Fig2C.png', bbox_inches='tight', dpi=300)


In [ ]:
# all models
for label, (df, col, _) in models.items():
    print(label)


In [ ]:
# percentage of queries within 1 / 3 / 5 points of the truth, baseline vs +MCP (same data as Fig 2C)
rows = {}
for label, (df, col, _) in models.items():
    d = df[col].dropna()
    rows[label] = {'queries (N)': len(d),
                   'mean (answer - truth)': round(d.mean(), 2),
                   }
    rows[label].update({f'% queries within {t} pts': round(100 * (d.abs() <= t).mean(), 2) for t in (1, 3, 5, 10)})
    rows[label]['queries within 3 pts (N)'] = int((d.abs() <= 3).sum())
within = pd.DataFrame(rows).T
within['queries (N)'] = within['queries (N)'].astype(int)
within['queries within 3 pts (N)'] = within['queries within 3 pts (N)'].astype(int)
display(within)
within.to_csv('Output/baseline_versus_MCP_stats.csv')

In [ ]:
5713/6011, 5833/6011

In [ ]:
87/6011, 13/6011

### Fraction of queries within x points of the truth, for the MCP models

In [ ]:
mcp_models = {
    'gpt-oss-20b+MCP':  (gpt_oss_20b_with_mcp,  'delta_gpt_oss_20b_with_mcp',  'darkgreen'),
    'gpt-oss-120b+MCP': (gpt_oss_120b_with_mcp, 'delta_gpt_oss_120b_with_mcp', 'firebrick'),
}

# x-values, evenly spaced 401 points from 0-10
# we are only testing within 10 points of the truth
thresholds = np.linspace(0, 10, 401)

# some distances to report in a summary table
# e.g. fraction of queries within 1 point of the truth, within 2 points of the truth etc.
marks = [1, 2, 3, 5, 10]

# just one axis figure
fig, ax = plt.subplots(figsize=(6, 5))
# empty dict to create table rows
rows = {}
for label, (df, col, color) in mcp_models.items():
    # get absolute deltas
    err = df[col].abs().dropna().to_numpy()
    # get frac of queries with error of atmost t, for each threshold t
    frac = np.array([(err <= t).mean() for t in thresholds])
    ax.plot(thresholds, frac * 100, color=color, lw=2, label=f'{label} (n={len(err)})')
    rows[label] = {f'<= {m} pts': f'{(err <= m).mean() * 100:.1f}%' for m in marks}

ax.axvline(3, color='gray', linestyle=':', lw=1)
ax.axhline(95, color='gray', linestyle=':', lw=1)
ax.set_xlim(0, 10)
# both models are above 80% almost immediately
ax.set_ylim(80, 100.5)
ax.set_xlabel('|Average answer - truth| (percentage points)', fontweight='bold')
ax.set_ylabel('% of queries within distance', fontweight='bold')
ax.legend(loc='lower right', fontsize=8)
plt.tight_layout()
plt.savefig('figs/Fig2D.png', bbox_inches='tight', dpi=300)

pd.DataFrame(rows).T


### Fraction of queries within x points of the truth broken down by intent, for the MCP models

In [ ]:
# fraction of queries within x points of the truth, per intent, for the MCP models
intents = sorted(gpt_oss_20b_with_mcp['intent'].unique())
ncols = 3
nrows = int(np.ceil(len(intents) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3.2 * nrows), sharex=True, sharey=True, squeeze=False)

rows = {}
for ax, intent in zip(axes.flat, intents):
    for label, (df, col, color) in mcp_models.items():
        err = df.loc[df['intent'] == intent, col].abs().dropna().to_numpy()
        frac = np.array([(err <= t).mean() for t in thresholds])
        ax.plot(thresholds, frac * 100, color=color, lw=2, label=label)
        rows[(intent, label)] = {'n': len(err), **{f'<= {m} pts': f'{(err <= m).mean() * 100:.1f}%' for m in marks}}
    ax.axvline(3, color='gray', linestyle=':', lw=1)
    ax.axhline(95, color='gray', linestyle=':', lw=1)
    ax.set_title(intent, fontsize=8, fontweight='bold')
    ax.tick_params(labelsize=8)
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 101)
for ax in axes.flat[len(intents):]:
    ax.set_visible(False)

fig.supxlabel('|Average answer - truth| (percentage points)', fontweight='bold')
fig.supylabel('% of queries within distance', fontweight='bold')
axes.flat[0].legend(loc='lower right', fontsize=8)
plt.tight_layout()
plt.savefig('figs/Fig2E.png', bbox_inches='tight', dpi=300)

pd.DataFrame(rows).T

pd.DataFrame(rows).T.to_csv('Output/errors_cumulative.by_intent.csv')

### assess spread across 3 tries per queries for the MCP models
- spread = max - min of the numeric values
- only numeric tries are used considered
- failed/ambiguous tries are ignored

In [ ]:
def add_response_range(df):
    # keep only numeric values (drop failed/ambiguous tries)
    vals = pd.concat(
        [pd.to_numeric(df[f'response_value_{i}'].where(df[f'response_status_{i}'] == 'numeric'), errors='coerce')
         for i in (1, 2, 3)],
        axis=1)
    # a query needs >= 2 numeric tries to have a spread; otherwise it is NaN and dropped
    df['response_range'] = (vals.max(axis=1) - vals.min(axis=1)).where(vals.notna().sum(axis=1) >= 2)

for df, _, _ in mcp_models.values():
    add_response_range(df)

In [ ]:
mcp_models['gpt-oss-20b+MCP'][0].head(n=3)

In [ ]:
# long format: one row per try (3 per query) for each MCP model
tries = pd.concat([
    pd.DataFrame({
        'model': label,
        'question': df['questions'],
        'intent': df['intent'],
        'try': i,
        'status': df[f'response_status_{i}'],
        'value': df[f'response_value_{i}'],
        'response': df[f'response_full_{i}'].astype(str),
    })
    for label, (df, _, _) in mcp_models.items() for i in (1, 2, 3)
], ignore_index=True)


In [ ]:
tries.groupby('model').agg('count')

In [ ]:
tries.shape

In [ ]:
18033*2

In [ ]:
tries.head(n=3)

In [ ]:
# one table: number of tries per status with % of tries in brackets, plus totals
counts = tries.groupby(['status', 'model']).size().unstack('model', fill_value=0)
total = counts.sum()
table = counts.astype(str) + ' (' + (100 * counts / total).round(1).astype(str) + '%)'
table.loc['total tries'] = total.astype(str)
table.loc['total queries'] = tries.groupby('model')['question'].nunique().astype(str)

# number of queries by how many of their 3 tries failed
n_failed = (tries.assign(is_failed=tries['status'] == 'failed')
            .groupby(['model', 'question'])['is_failed'].sum()
            .groupby('model').value_counts().unstack('model', fill_value=0))
for k, lab in [(3, 'queries with all 3 tries failed'), (2, 'queries with 2 tries failed'), (1, 'queries with 1 try failed')]:
    table.loc[lab] = n_failed.loc[k].astype(str) if k in n_failed.index else '0'

display(table)
table.to_csv('Output/mcp_try_status_summary.csv', index_label='status')

In [ ]:
# example rows: 2 per status per model, printed in full
for status in ['numeric', 'failed', 'ambiguous']:
    print(f'\n=== {status} ===')
    for model, g in tries[tries['status'] == status].groupby('model'):
        print(f'-- {model} ({len(g)} tries)')
        for _, r in g.head(1).iterrows():
            print(f'intent: {r["intent"]} | try: {r["try"]} | value: {r["value"]}')
            print(f'question: {r["question"]}')
            # responses are huge, commenting this out
            # uncomment to see all
            # print(f'response: {r["response"]}\n')

### Investigate examples of failed or ambiguous queries

In [ ]:
# failed / ambiguous tries per model, with examples of each error type
# each query was tried 3x, and each try has a status (numeric / failed / ambiguous)
def answer_text(resp):
    # pull the model's answer out of the raw response string
    try:
        return ast.literal_eval(resp)['response']['answer']
    except Exception:
        return resp

def error_kind(status, resp):
    if status == 'failed':
        if 'Internal Server Error' in resp:
            return 'HTTP 500 Internal Server Error'
        return 'read timeout' if 'timed out' in resp else 'other'
    # ambiguous: the model asked a clarifying question instead of answering
    a = answer_text(resp).lower()
    if 'project id' in a or 'gene symbol' in a:
        return 'asks to confirm project / gene'
    if any(k in a for k in ['copy', 'cnv', 'mutation', 'deletion']):
        return 'asks which alteration type (CNV vs SSM)'
    return 'empty / non-answer'


In [ ]:
# one row per failed or ambiguous try
recs = []
for label, (df, _, _) in mcp_models.items():
    for i in (1, 2, 3):
        bad = df[df[f'response_status_{i}'].isin(['failed', 'ambiguous'])]
        for q, intent, st, r in zip(bad['questions'], bad['intent'], bad[f'response_status_{i}'], bad[f'response_full_{i}']):
            r = str(r)
            recs.append(dict(model=label, intent=intent, status=st, kind=error_kind(st, r), question=q,
                             text=r if st == 'failed' else answer_text(r)))
errs = pd.DataFrame(recs)

In [ ]:
errs.shape

In [ ]:
errs[errs['model'] == 'gpt-oss-120b+MCP'].head(n=3)

In [ ]:
errs.groupby(['model', 'kind']).agg('count')

In [ ]:
errs[errs['model'] == 'gpt-oss-120b+MCP'].head(n=3).loc[1925,:]['question']

In [ ]:
gpt_oss_120b_with_mcp[gpt_oss_120b_with_mcp['questions'] == 'What fraction of cases have either simple somatic mutations or copy number variants in ALK in Prostate Adenocarcinoma TCGA-PRAD project in the genomic data commons?']

## number of queries with identical variant frequencies across triplicates
- based on response_range column, which includes queries with atleast 2 numeric tries

In [ ]:
gpt_oss_20b_with_mcp.head(n=3)

In [ ]:
gpt_oss_20b_with_mcp[gpt_oss_20b_with_mcp['response_range'] == 0].shape

In [ ]:
# ambiguous or failed
gpt_oss_20b_with_mcp[gpt_oss_20b_with_mcp['response_range'].isna()].shape

In [ ]:
 gpt_oss_20b_with_mcp[gpt_oss_20b_with_mcp['response_range'].isna()].head(n=3)

In [ ]:
# total number of queries with atleast 2 numeric tries, ignoring failed or ambiguous queries
6011-gpt_oss_20b_with_mcp[gpt_oss_20b_with_mcp['response_range'].isna()].shape[0]

In [ ]:
# percentage of queries with atleast 2 numeric tries, where variant frequency values are identical
gpt_oss_20b_with_mcp[gpt_oss_20b_with_mcp['response_range'] == 0].shape[0]/ (6011-gpt_oss_20b_with_mcp[gpt_oss_20b_with_mcp['response_range'].isna()].shape[0])

In [ ]:
gpt_oss_120b_with_mcp.head(n=3)

In [ ]:
gpt_oss_120b_with_mcp[gpt_oss_120b_with_mcp['response_range'] == 0].shape

In [ ]:
# total number of queries with atleast 2 numeric tries, ignoring failed or ambiguous queries
6011-gpt_oss_120b_with_mcp[gpt_oss_120b_with_mcp['response_range'].isna()].shape[0]

In [ ]:
5261/6011

In [ ]:
 gpt_oss_120b_with_mcp[gpt_oss_120b_with_mcp['response_range'].isna()].head(n=3)

## Fig 3A

In [ ]:
# stacked bars of how many queries fall in each spread bin
# only queries with >= 2 numeric tries have a spread (response_range), so n differs by model
edges  = [0, 1e-9, 1, 3, 10, np.inf]
bin_labels = ['identical (0)', '0-1 pt', '1-3 pts', '3-10 pts', '> 10 pts']
bin_colors = cm.YlOrRd(np.linspace(0.12, 0.85, len(bin_labels)))

panels = ['All intents'] + intents
fig, axes = plt.subplots(1, len(panels), figsize=(2.6 * len(panels), 5), sharey=True)

for ax, name in zip(axes, panels):
    for k, (label, (df, col, color)) in enumerate(mcp_models.items()):
        d = df if name == 'All intents' else df[df['intent'] == name]
        # dropna keeps queries with >= 2 numeric tries
        rng = d['response_range'].dropna()
        # 'identical' = range of exactly 0; remaining bins are (lo, hi]
        counts = pd.cut(rng, bins=[-1] + edges[1:], labels=bin_labels).value_counts(normalize=True).reindex(bin_labels) * 100
        bottom = 0
        for lab, c in zip(bin_labels, bin_colors):
            ax.bar(k, counts[lab], bottom=bottom, color=c, edgecolor='black', linewidth=0.4, width=0.7, label=lab)
            bottom += counts[lab]
        ax.text(k, 101, f'n={len(rng)}', ha='center', va='bottom', fontsize=6)
    ax.set_xticks(range(len(mcp_models)))
    ax.set_xticklabels(list(mcp_models), fontsize=8, rotation=90, fontweight='bold')
    ax.set_title(name, fontsize=8, fontweight='bold', pad=14)
    ax.set_ylim(0, 100)
    ax.tick_params(axis='y', labelsize=8)

axes[0].set_ylabel('% of queries', fontweight='bold')
handles = [Patch(facecolor=c, edgecolor='black', label=l) for l, c in zip(bin_labels, bin_colors)]
axes[-1].legend(handles=handles, title='Spread of numeric tries\n(max - min)', title_fontsize=8, bbox_to_anchor=(1, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.savefig('figs/Fig3A.png', bbox_inches='tight', dpi=300)

### Investigate spread of responses on queries that returned a numeric answer in all 3 tries in both models
- Keep a query only if both models returned a numeric answer on all 3 tries
- That gives 4,263 queries, the same ones for each model
- measure consistency within each model: On those 4,263 queries, take each model separately and ask whether its own three answers were equal (max-min or spread = 0)
- The result was 2,718 of 4,263 (63.8%) for 20b+MCP and 3,755 of 4,263 (88.1%) for 120b+MCP

In [ ]:
# number of numeric tries per query (0-3)
def n_numeric(df):
    return sum((df[f'response_status_{i}'] == 'numeric').astype(int) for i in (1, 2, 3))

tab = pd.DataFrame({label: n_numeric(df).value_counts().reindex([0, 1, 2, 3], fill_value=0)
                    for label, (df, _, _) in mcp_models.items()})
tab.index.name = 'numeric tries (of 3)'
print(tab)

# keep only queries where every model has all 3 numeric tries, so both are compared on the same queries
# (matched on question text, because the two files have the rows in a different order)
keep = set.intersection(*[set(df.loc[n_numeric(df) == 3, 'questions']) for df, _, _ in mcp_models.values()])
print(f'\nqueries with 3 numeric tries in every model: {len(keep)}')

# plot same stacked bars as above, restricted to those queries (so n is identical for both models)
edges  = [0, 1e-9, 1, 3, 10, np.inf]
bin_labels = ['identical (0)', '0-1 pt', '1-3 pts', '3-10 pts', '> 10 pts']
bin_colors = cm.YlOrRd(np.linspace(0.12, 0.85, len(bin_labels)))

panels = ['All intents'] + intents
fig, axes = plt.subplots(1, len(panels), figsize=(2.6 * len(panels), 5), sharey=True)

for ax, name in zip(axes, panels):
    for k, (label, (df, col, color)) in enumerate(mcp_models.items()):
        d = df[df['questions'].isin(keep)]
        d = d if name == 'All intents' else d[d['intent'] == name]
        rng = d['response_range'].dropna()
        # 'identical' = range of exactly 0; remaining bins are (lo, hi]
        counts = pd.cut(rng, bins=[-1] + edges[1:], labels=bin_labels).value_counts(normalize=True).reindex(bin_labels) * 100
        bottom = 0
        for lab, c in zip(bin_labels, bin_colors):
            ax.bar(k, counts[lab], bottom=bottom, color=c, edgecolor='black', linewidth=0.4, width=0.7, label=lab)
            bottom += counts[lab]
        ax.text(k, 101, f'n={len(rng)}', ha='center', va='bottom', fontsize=6)
    ax.set_xticks(range(len(mcp_models)))
    ax.set_xticklabels(list(mcp_models), fontsize=8, rotation=90, fontweight='bold')
    ax.set_title(name, fontsize=8, fontweight='bold', pad=14)
    ax.set_ylim(0, 100)
    ax.tick_params(axis='y', labelsize=8)

axes[0].set_ylabel('% of queries', fontweight='bold')
handles = [Patch(facecolor=c, edgecolor='black', label=l) for l, c in zip(bin_labels, bin_colors)]
axes[-1].legend(handles=handles, title='Spread of 3 tries\n(max - min)', title_fontsize=8, bbox_to_anchor=(1, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.savefig('figs/S9.png', bbox_inches='tight', dpi=300)

### Investigate ssm_frequency and freq_cnv_loss_or_gain_comb errors
- inspect errors, present representative example queries
- add plots with error trends tested more systematically

In [ ]:
# ssm_frequency queries from gpt-oss-120b+MCP where the average answer is not equal to the truth
ssm = gpt_oss_120b_with_mcp[gpt_oss_120b_with_mcp['intent'] == 'ssm_frequency'].copy()
ssm['wrong'] = ssm['answer_average'].round(2) != ssm['truth'].round(2)
ssm_wrong = ssm[ssm['wrong']].copy()
print(f'ssm_frequency queries: {len(ssm)}, wrong: {len(ssm_wrong)}')

# did the model write the mutation as 'p.G13D' instead of 'G13D' in the tool call?
# in the response the tool call looks like: "aa_change": "p.G13D"
# so we take the text after the last "aa_change" and check if it starts with 'p.'
def used_p(response):
    text = str(response)
    if '"aa_change": "' not in text:
        return False
    after_last_aa_change = text.rsplit('"aa_change": "', 1)[-1]
    return after_last_aa_change.startswith('p.')

# number of tries (0 to 3) that used 'p.'
ssm_wrong['tries_with_p'] = (ssm_wrong['response_full_1'].map(used_p).astype(int)
                             + ssm_wrong['response_full_2'].map(used_p).astype(int)
                             + ssm_wrong['response_full_3'].map(used_p).astype(int))
ssm_wrong['used_p'] = ssm_wrong['tries_with_p'] > 0

# how far is the average answer from the truth?
ssm_wrong['answer - truth'] = (ssm_wrong['answer_average'] - ssm_wrong['truth']).round(2)
ssm_wrong['error > 1 pt'] = ssm_wrong['answer - truth'].abs() > 1

# count wrong queries: used 'p.' or not, and whether the error is larger than 1 point
print(pd.crosstab(ssm_wrong['used_p'], ssm_wrong['error > 1 pt'], margins=True))

# save to CSV
ssm_wrong.to_csv('Output/ssm_frequency_wrong_rows.csv', index=False)
ssm_wrong[['questions', 'truth', 'answer_average', 'answer - truth', 'tries_with_p']].head(10)

In [ ]:
ssm_wrong['tries_with_p'].value_counts()

In [ ]:
ssm_wrong[ssm_wrong['tries_with_p'] == 0].head()

In [ ]:
# all ssm_frequency queries (not only the wrong ones) from gpt-oss-120b+MCP
# uses ssm and used_p from the cell above
ssm['tries_with_p'] = (ssm['response_full_1'].map(used_p).astype(int)
                       + ssm['response_full_2'].map(used_p).astype(int)
                       + ssm['response_full_3'].map(used_p).astype(int))
ssm['answer - truth'] = ssm['answer_average'] - ssm['truth']

# one dot per query; x = number of tries (0-3) that used 'p.', y = average answer - truth
# dots are shifted sideways a little (jitter) so they do not sit on top of each other
rng = np.random.default_rng(0)
x = ssm['tries_with_p'] + rng.uniform(-0.2, 0.2, len(ssm))

fig, ax = plt.subplots(figsize=(6, 5))
ax.axhline(0, color='black', linestyle='--', lw=1)
ax.scatter(x, ssm['answer - truth'], color='firebrick', edgecolor='black', linewidth=0.4, s=25, alpha=0.7)

# the number of queries in each group is written on top
for k in range(4):
    group = ssm[ssm['tries_with_p'] == k]
    ax.text(k, ax.get_ylim()[1], f'n={len(group)}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(range(4))
ax.set_xlabel('Number of tries (of 3) that used p.', fontweight='bold')
ax.set_ylabel('Average answer - truth (%)', fontweight='bold')
ax.set_title('gpt-oss-120b+MCP, ssm_frequency', fontweight='bold', pad=18)
plt.tight_layout()
# plt.savefig('figs/Fig4A.png', bbox_inches='tight', dpi=300)

# the same in numbers
ssm.groupby('tries_with_p')['answer - truth'].describe().round(2)

In [ ]:
75+29+29+11

In [ ]:
29+29+11

In [ ]:
ssm.shape

In [ ]:
ssm_wrong.shape

In [ ]:
86/144

In [ ]:
1-(86/144)

### Examples of API calls with and without the p.

In [ ]:
import requests
import json

In [ ]:
def get_ssm_id(gene, mutation):
    ssm_id_endpt = "https://api.gdc.cancer.gov/ssms"
    fields = ["mutation_type"]
    fields = ",".join(fields)
    expand = ["consequence.transcript"]
    filters = {
        "op": "=",
        "content": {"field": "gene_aa_change", "value": "[gene][mutation]"},
    }
    filters["content"]["value"] = gene + " " + mutation
    # print('filters {}'.format(filters))
    params = {
        "filters": json.dumps(filters),
        "fields": fields,
        "expand": expand,
        "size": 10,
    }
    ssm_id = None
    try:
        response = requests.get(ssm_id_endpt, params=params)
        response_json = json.loads(response.content)
        ssm_id = response_json["data"]["hits"][0]["id"]
    except Exception as e:
        print("unable to execute GDC API request {}".format(str(e)))
    return ssm_id

In [ ]:
def get_ssm_id2(gene, hgvsp):
    ssm_id_endpt = "https://api.gdc.cancer.gov/ssms"
    filters = {
        "op": "and",
        "content": [
            {"op": "in", "content": {"field": "consequence.transcript.gene.symbol", "value": [gene]}},
            {"op": "in", "content": {"field": "consequence.transcript.annotation.hgvsp_short", "value": [hgvsp]}},
        ],
    }
    params = {
        "filters": json.dumps(filters),
        "fields": "mutation_type",
        "expand": ["consequence.transcript"],
        "size": 10,
    }
    ssm_id = None
    try:
        response = requests.get(ssm_id_endpt, params=params)
        response.raise_for_status()
        ssm_id = response.json()["data"]["hits"][0]["id"]
    except Exception as e:
        print("unable to execute GDC API request {}".format(e))
    return ssm_id

In [ ]:
get_ssm_id('BRAF', 'V600E')

In [ ]:
get_ssm_id('BRAF', 'p.V600E')

In [ ]:
get_ssm_id2('BRAF', 'p.V640E')

In [ ]:
get_ssm_id2('BRAF', 'p.V600E')

In [ ]:
get_ssm_id('KRAS', 'G13D')

In [ ]:
get_ssm_id('KRAS', 'p.G13D')

In [ ]:
get_ssm_id2('KRAS', 'p.G13D')

### ssm frequency trends for both models
- plot gpt-oss-20b+MCP and gpt-oss-120b+MCP side-by-side

In [ ]:
# same plot as ssm_frequency queries with p., but plot both models
# each model is shifted slightly left/right of the tick so the two sets of dots do not sit on top of each other
ssm_models = {'gpt-oss-120b+MCP': (gpt_oss_120b_with_mcp, 'firebrick', -0.15),
              'gpt-oss-20b+MCP': (gpt_oss_20b_with_mcp, 'steelblue', 0.15)}

fig, ax = plt.subplots(figsize=(7, 5))
ax.axhline(0, color='black', linestyle='--', lw=1)
rng = np.random.default_rng(0)
for label, (df, color, shift) in ssm_models.items():
    d = df[df['intent'] == 'ssm_frequency'].copy()
    # the base x-position: an integer from 0 to 3: how many of the 3 tries for that query used p.
    d['tries_with_p'] = sum(d[f'response_full_{i}'].map(used_p).astype(int) for i in (1, 2, 3))
    d['answer - truth'] = d['answer_average'] - d['truth']
    # compute horizontal position of each dot on the plot
    # shift separates the two models (-0.15 for gpt-oss-120b+MCP and +0.15 for gpt-oss-20b+MCP)
    # rng.uniform(-0.1, 0.1, len(d)) is jitter to clearly sep the points: a random offset between -0.1 and +0.1 to each dot, with one value per row
    # The seed is fixed (default_rng(0)), so the jitter is the same every run.
    # the jitter (±0.1) is smaller than the shift (±0.15)
    # each model's cloud therefore spans 0.2 around its center, which is 0.3 away from the other model's center
    # Note: the x-position is only a visual offset and carries no data, only the tick group (0–3) means anything.
    x = d['tries_with_p'] + shift + rng.uniform(-0.1, 0.1, len(d))
    ax.scatter(x, d['answer - truth'], color=color, edgecolor='black', linewidth=0.4, s=25, alpha=0.7, label=label)
    # the number of queries in each group is written on top, in the model's colour
    for k in range(4):
        ax.text(k + shift, 1.0, f"n={(d['tries_with_p'] == k).sum()}", ha='center', va='bottom', fontsize=8,
                color=color, fontweight='bold', transform=ax.get_xaxis_transform())

ax.set_xticks(range(4))
ax.set_xlim(-0.5, 3.5)  # leave room so the n labels of the last group stay inside the plot
ax.set_xlabel('Number of tries (of 3) that used p.', fontweight='bold')
ax.set_ylabel('Average answer - truth (%)', fontweight='bold')
ax.set_title('ssm_frequency', fontweight='bold', pad=18)
ax.legend(fontsize=8, loc='lower left')
plt.tight_layout()
plt.savefig('figs/Fig4A.png', bbox_inches='tight', dpi=300)

In [ ]:
d.shape

In [ ]:
75+29+29+11

### Investigate freq_cnv_loss_gain_comb errors
- side-by-side comparison of gpt-oss-120b+MCP and gpt-oss-20b+MCP

In [ ]:
# all freq_cnv_loss_or_gain_comb queries (co-occurrence of two CNV events), both models + MCP
# question: "What is the co-occurence frequency of heterozygous deletions in PTEN and CDKN2A in <project>?"
# correct answer = cases with deletions in borh genes / all cases with CNV data in the project
# the model has to ask the tool for the size of the denominator, via get_case_set_size
# the correct denominator is the case set called 'Cases-Data-(cnv)-...'; the model sometimes picks
# a different set, e.g. 'Cases-Union-...' or 'Cases-Intersect-...'
import ast, json

def case_sets_sized(response):
    # every case set whose size is computed
    try:
        steps = ast.literal_eval(str(response))['steps']
    except Exception:
        return []
    return [json.loads(s['args'])['case_set_id'] for s in steps
            if s.get('type') == 'tool_call' and s.get('tool') == 'get_case_set_size']

def used_wrong_denominator(response):
    # wrong = a denominator which is not the get_cases_with_available_genomic_data
    sets = case_sets_sized(response)
    return len(sets) > 0 and not any(s.startswith('Cases-Data-(cnv)') for s in sets)

# one dot per query; x = number of tries (0-3) that used a wrong denominator, y = average answer - truth
cnv_models = {'gpt-oss-120b+MCP': (gpt_oss_120b_with_mcp, 'firebrick', -0.15),
              'gpt-oss-20b+MCP': (gpt_oss_20b_with_mcp, 'steelblue', 0.15)}

fig, ax = plt.subplots(figsize=(7, 5))
ax.axhline(0, color='black', linestyle='--', lw=1)
rng = np.random.default_rng(0)
for label, (df, color, shift) in cnv_models.items():
    d = df[df['intent'] == 'freq_cnv_loss_or_gain_comb'].copy()
    d['tries_wrong_den'] = sum(d[f'response_full_{i}'].map(used_wrong_denominator).astype(int) for i in (1, 2, 3))
    d['answer - truth'] = d['answer_average'] - d['truth']
    x = d['tries_wrong_den'] + shift + rng.uniform(-0.1, 0.1, len(d))
    ax.scatter(x, d['answer - truth'], color=color, edgecolor='black', linewidth=0.4, s=25, alpha=0.7, label=label)
    for k in range(4):
        ax.text(k + shift, 1.0, f"n={(d['tries_wrong_den'] == k).sum()}", ha='center', va='bottom', fontsize=8,
                color=color, fontweight='bold', transform=ax.get_xaxis_transform())
    print(label)
    print(d.groupby('tries_wrong_den')['answer - truth'].describe().round(2))
    cnv_comb = d 

ax.set_xticks(range(4))
ax.set_xlim(-0.5, 3.5)
ax.set_xlabel('Number of tries (of 3) that used a wrong denominator', fontweight='bold')
ax.set_ylabel('Average answer - truth (%)', fontweight='bold')
ax.set_title('freq_cnv_loss_or_gain_comb', fontweight='bold', pad=18)
ax.legend(fontsize=8, loc='upper left')
plt.tight_layout()
# plt.savefig('figs/Fig4C.png', bbox_inches='tight', dpi=300)

In [ ]:
# fig. 4B
def any_cnv(case_set):
    return re.search(r'Cases-CNV-\([^)]*\)-\(None\)', case_set) is not None

def denominator_type(response):
    sets = case_sets_sized(response)
    if not sets:
        return None  # the model never asked for a size (e.g. the try failed)
    if any(s.startswith('Cases-Data-(cnv)') for s in sets):
        return 'all cases with CNV data (correct)'
    if any(s.startswith('Cases-Union-') for s in sets):
        return 'union of the 2 case sets with CNVs'
    if any(s.startswith('Cases-Intersect-') and any_cnv(s) for s in sets):
        return 'intersection of the two case sets with CNVs'
    return 'other'

types = ['all cases with CNV data (correct)', 'union of the 2 case sets with CNVs',
         'intersection of the two case sets with CNVs', 'other']

types_all = types + ['unparsable']

# check if logs are truncated
def is_truncated(response):
    return len(str(response)) >= 32767

def denominator_type_all(response):
    t = denominator_type(response)
    return 'unparsable' if t is None else t



fig, ax = plt.subplots(figsize=(10, 5.5))
ax.axhline(0, color='black', linestyle='--', lw=1)
rng = np.random.default_rng(0)
plotted, n_excluded = {}, 0
for label, (df, color, shift) in cnv_models.items():
    d = df[df['intent'] == 'freq_cnv_loss_or_gain_comb']
    tries = pd.concat([pd.DataFrame({'type': d[f'response_full_{i}'].map(
                                         lambda r: 'unparsable' if denominator_type(r) is None else denominator_type(r)),
                                     'truncated': d[f'response_full_{i}'].map(is_truncated),
                                     'answer': pd.to_numeric(d[f'response_value_{i}'], errors='coerce'),
                                     'truth': d['truth']}) for i in (1, 2, 3)])
    assert len(tries) == 3 * len(d)
    # a truncated log does not have enough info to parse (i.e 'unparsable')
    # in this case all of them have a numeric answer
    assert (tries.loc[tries['truncated'], 'type'] == 'unparsable').all()
    n_excluded += tries['truncated'].sum()
    tries = tries[~tries['truncated']].copy()
    tries['answer - truth'] = tries['answer'] - tries['truth']
    tries['x'] = tries['type'].map(types_all.index) + shift + rng.uniform(-0.1, 0.1, len(tries))
    plotted[label] = tries
    has = tries['answer - truth'].notna()
    ax.scatter(tries.loc[has, 'x'], tries.loc[has, 'answer - truth'], color=color, edgecolor='black',
               linewidth=0.4, s=25, alpha=0.7, label=label)
    for k, t in enumerate(types_all):
        ax.text(k + shift, 1.0, f"n={(tries['type'] == t).sum()}", ha='center', va='bottom', fontsize=8,
                color=color, fontweight='bold', transform=ax.get_xaxis_transform())

# strip for tries without a numeric answer
lo, hi = ax.get_ylim()
strip_y = lo - 0.07 * (hi - lo)
for label, (df, color, shift) in cnv_models.items():
    tries = plotted[label]
    na = tries[tries['answer - truth'].isna()]
    ax.scatter(na['x'], np.full(len(na), strip_y), marker='x', color=color, s=20, linewidth=0.8, alpha=0.7,
               clip_on=False, label=f'{label}: no numeric answer (n={len(na)})')
ax.scatter([], [], color='none', label=f'N={n_excluded} tries could not be confidently\n'
                                       'assigned to a category due to truncated\n'
                                       'reasoning logs and were excluded\nfrom the plot')
ax.set_ylim(strip_y - 0.04 * (hi - lo), hi)
ax.axhline(lo - 0.035 * (hi - lo), color='grey', lw=0.6)

ax.set_xticks(range(len(types_all)))
ax.set_xticklabels([t.replace(' (', '\n(').replace(', any', ',\nany').replace('of the', 'of the\n').replace(' / ', ' /\n')
                    for t in types_all], fontsize=8)
ax.set_xlim(-0.5, len(types_all) - 0.5)
ax.set_xlabel('Denominator used', fontweight='bold')
ax.set_ylabel('Answer of each try - truth (%)', fontweight='bold')
ax.set_title('freq_cnv_loss_or_gain_comb', fontweight='bold', pad=18)
ax.legend(fontsize=8, loc='upper right')
plt.tight_layout()
plt.savefig('figs/Fig4B.png', bbox_inches='tight', dpi=300)

In [ ]:
# example queries with wrong denominator used
tries_all = []
for label, (df, color, shift) in cnv_models.items():
    d = df[df['intent'] == 'freq_cnv_loss_or_gain_comb']
    for row, r in d.iterrows():
        for i in (1, 2, 3):
            t = denominator_type(r[f'response_full_{i}'])
            answer = pd.to_numeric(r[f'response_value_{i}'], errors='coerce')
            tries_all.append({'model': label, 'denominator used': 'unparsable' if t is None else t,
                              'row': row, 'try': i, 'query': r['questions'], 'truth': r['truth'],
                              'answer of try': answer, 'answer - truth': round(answer - r['truth'], 2)})
tries_all = pd.DataFrame(tries_all)
pd.set_option('display.max_colwidth', None)

# union / intersection denominators: first 3 tries with a numeric answer per model and type
ui = tries_all[tries_all['denominator used'].isin(['union of the 2 case sets with CNVs',
                                                   'intersection of the two case sets with CNVs'])
               & tries_all['answer of try'].notna()]
print(ui.groupby(['model', 'denominator used']).size())
display(ui.groupby(['model', 'denominator used']).head(3).sort_values(['model', 'denominator used', 'row']))

# unparsable tries that do have a numeric answer
# response_full of these is cut at 32,767 characters, so the denominator cannot be found;
# the answer is read from response_value, which comes from the start of the response
unp = tries_all[(tries_all['denominator used'] == 'unparsable') & tries_all['answer of try'].notna()]
print(f'\nunparsable tries with a numeric answer: {len(unp)}')
display(unp)